# 02 · Catálogo da Vulto: raio-x

Fonte: `catalogo_produtos.csv` (as 5 abas de `catalogo_vulto_2026_REVISADO_final.xlsx` juntas, geradas por `padronizar.py`).

Objetivo: **entender** o catálogo, sem decidir nem corrigir nada. As mesmas 4 perguntas das vendas.

**Como é o Excel original** (só olhar; nunca salvar pelo Excel): uma aba por coleção (VERAO 25, inverno25, Verão 26, Atemporais, PLANILHA ANTIGA NAO USAR), cada uma montada por uma pessoa, com cabeçalho em linhas diferentes e nomes de coluna diferentes. Descontinuado é marcado com **texto tachado ou linha vermelha** e "revisar preço" com **fundo amarelo** (o `padronizar.py` transformou essas marcações nas colunas `descontinuado` e `revisar_preco`). A aba "LEIA ISSO - REGRAS" (Marina, Compras) diz: peça em duas abas vale o preço da coleção **mais nova**; custo = **último** custo de nota; código pode ser **reaproveitado** quando o modelo sai de linha; o CD usa o código curto e o fiscal "digita do jeito dele".

> Antes de commitar: *Kernel → Restart & Clear Output*.

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
catalogo = pd.read_csv(STAGING / "catalogo_produtos.csv", dtype=str)
itens = pd.read_csv(STAGING / "banco_itens_venda.csv", dtype=str)
estoque = pd.read_csv(STAGING / "banco_estoque.csv", dtype=str)

## 1. O que é cada linha?

Um SKU (produto + cor + tamanho) numa coleção (`aba`).

In [ ]:
print("linhas:", len(catalogo), "| SKUs distintos:", catalogo["sku"].nunique())
display(catalogo["aba"].value_counts())
catalogo.head()

In [ ]:
# uma linha que não é produto: o título de uma coleção nova dentro da aba Verão 26
titulo = catalogo.index[catalogo["sku"].str.startswith("INVERNO 26")][0]
print("produtos depois do título, na mesma aba:", (catalogo.loc[titulo:, "aba"] == "Verão 26").sum() - 1)
catalogo.loc[titulo - 1:titulo + 3, ["linha_origem", "aba", "sku", "nome", "preco"]]

## 2. O que significa cada coluna?

In [ ]:
for coluna in ["categoria", "cor", "tamanho"]:
    print(coluna, "| grafias:", catalogo[coluna].nunique(), "|", catalogo[coluna].value_counts().head(8).to_dict())

In [ ]:
print("preço vazio:", catalogo["preco"].isna().sum(), "| custo vazio por aba:", catalogo.groupby("aba")["custo"].apply(lambda s: s.isna().sum()).to_dict())
print("EAN vazio:", catalogo["ean"].isna().sum(),
      "| EAN repetido em SKUs diferentes:", catalogo.dropna(subset=["ean"]).drop_duplicates(["sku", "ean"])["ean"].duplicated().sum())
pd.crosstab(catalogo["aba"], catalogo["descontinuado"].fillna("(sem marcação)"))

## 3. Que problemas aparecem?

In [ ]:
# cada aba escreve o SKU de um jeito
formato = catalogo["sku"].map(lambda s: "longo (VLT-...)" if s.startswith("VLT-") else "com espaço" if " " in s else "curto")
pd.crosstab(catalogo["aba"], formato)

In [ ]:
# SKU em duas abas: com preço diferente?
repetidos = catalogo[catalogo["sku"].duplicated(keep=False)]
por_sku = repetidos.groupby("sku")
print("SKUs em 2 abas:", por_sku.ngroups, "| com preço diferente:", (por_sku["preco"].nunique() > 1).sum())
display(por_sku["aba"].apply(lambda s: " + ".join(sorted(s))).value_counts())
repetidos[repetidos["sku"].isin(["BER003B40", "ACE005GU"])][["aba", "sku", "nome", "preco", "descontinuado"]]

In [ ]:
import re

# Teste: o mesmo SKU escrito de 3 jeitos (VLT-CAM-004-BRA-M, CAM004BM, vlt cam 004 branco m) vira a mesma chave
# CAT + NNN + inicial da cor + tamanho. Serve só para medir; a regra de verdade é decisão da etapa 4.
def chave(sku):
    s = str(sku).upper().strip()
    longo = re.fullmatch(r"VLT[- ]([A-Z]{3})[- ](\d{3})[- ]([A-Z]+)[- ]?(.*)", s)
    if longo:
        return f"{longo[1]}{longo[2]}{longo[3][0]}{longo[4].replace(' ', '').replace('-', '')}"
    curto = re.fullmatch(r"([A-Z]{3})(\d{3})([A-Z])(.+)", s)
    if curto:
        return f"{curto[1]}{curto[2]}{curto[3]}{curto[4]}"
    return None

In [ ]:
# código reaproveitado: o mesmo modelo (CAT + NNN) com nomes diferentes
catalogo["modelo"] = catalogo["sku"].map(lambda s: (chave(s) or "")[:6])
nomes = catalogo[catalogo["modelo"] != ""].groupby("modelo")["nome"].apply(lambda s: s.str.upper().str.strip().nunique())
print("modelos com mais de um nome:", (nomes > 1).sum())
catalogo[catalogo["modelo"].isin(["JAQ001", "MOL001", "JAQ002"])].drop_duplicates(["modelo", "nome"])[["aba", "sku", "nome"]]

## 4. Como se liga às outras fontes?

O SKU liga o catálogo às vendas (`itens_venda`) e ao estoque.

In [ ]:
chaves_catalogo = set(catalogo["sku"].map(chave).dropna())
for nome, tabela in [("itens de venda", itens), ("estoque", estoque)]:
    print(f"{nome}: SKU escrito igual {tabela['sku'].isin(catalogo['sku']).mean():.0%}",
          f"| pela chave {tabela['sku'].map(chave).isin(chaves_catalogo).mean():.0%}")
# o que continua sem par no catálogo
itens.loc[~itens["sku"].map(chave).isin(chaves_catalogo), "sku"].value_counts().head(8)

## Desafios da revisão

In [ ]:
# os SKUs vendidos que não estão no catálogo: o modelo não existe, ou só faltam cores e tamanhos?
chaves_catalogo = set(catalogo["sku"].map(chave).dropna())
modelos_catalogo = {k[:6] for k in chaves_catalogo}
vendidos = itens["sku"].map(chave).dropna()
faltando = vendidos[~vendidos.isin(chaves_catalogo)].drop_duplicates()
print("variantes vendidas fora do catálogo:", len(faltando), "| o modelo existe no catálogo:", faltando.str[:6].isin(modelos_catalogo).sum())
# exemplo: vendeu a cor G do TRI002, mas o catálogo só tem a cor B
sorted(k for k in chaves_catalogo if k.startswith("TRI002")), sorted(k for k in faltando if k.startswith("TRI002"))

In [ ]:
# nos SKUs com preço diferente entre abas: a diferença é pequena ou grande? e qual coleção é mais cara?
precos = repetidos.assign(p=pd.to_numeric(repetidos["preco"])).groupby("sku")["p"].agg(["min", "max"])
precos = precos[precos["min"] != precos["max"]]
diferenca_pct = (precos["max"] - precos["min"]) / precos["min"] * 100
print("diferença em R$ (mediana):", (precos["max"] - precos["min"]).median(), "| em % (mediana):", round(diferenca_pct.median(), 1))
# ordem das coleções, da mais antiga para a mais nova (Atemporais × Verão 26 é discutível: Atemporais é revisada todo março)
ordem = {"PLANILHA ANTIGA NAO USAR": 0, "VERAO 25": 1, "inverno25": 2, "Atemporais": 3, "Verão 26": 4}
mais_nova = repetidos.assign(o=repetidos["aba"].map(ordem), p=pd.to_numeric(repetidos["preco"])).sort_values("o").groupby("sku")["p"].last()
print("a coleção mais nova é a mais cara em", (mais_nova.loc[precos.index] == precos["max"]).sum(), "de", len(precos))
pd.cut(diferenca_pct, [0, 5, 10, 20, 50, 1000]).value_counts().sort_index()

## Anotações para o dicionário de dados

**O que é cada linha**
- Um SKU (produto + cor + tamanho) numa coleção (`aba`). 1.174 linhas, 1.020 SKUs distintos.
- Abas: Verão 26 (449), PLANILHA ANTIGA NAO USAR (297), Atemporais (166), VERAO 25 (152), inverno25 (110).

**Colunas**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `aba` | coleção de onde veio | 177 produtos de **Inverno 26** estão dentro da aba Verão 26 (abaixo) |
| `sku` | código da variante | um formato por aba: VERAO 25 e antiga = longo `VLT-...`; inverno25 = com espaço; Verão 26 = curto `CAM016GP`; Atemporais = misturado |
| `nome` | nome do produto | o mesmo produto em maiúsculas e minúsculas (`Calça Jogger Destroyed` × `CALÇA JOGGER DESTROYED`) |
| `categoria` | tipo de peça | 57 grafias (`CALÇAS`, `Bottoms > Calça`, `Calças`, `CALÇA`) |
| `cor` | cor | 65 grafias (`BRANCO`, `White`, `branco`) |
| `tamanho` | tamanho | 23 valores (P/M/G/GG, numeração 36–46, U) |
| `preco` | preço de venda | 154 SKUs em duas abas, 128 com preço diferente |
| `custo` | custo unitário | 198 vazios (102 no Verão 26) |
| `ean` | código de barras | 25 vazios; 144 repetidos em SKUs diferentes |
| `descontinuado` | saiu de linha (tachado ou vermelho no Excel) | 165 sim; 93 sem marcação clara (só a célula do SKU em vermelho) |
| `revisar_preco` | fundo amarelo no Excel | 5 |

**Produtos de Inverno 26 dentro da aba Verão 26**
- **Fato:** a linha `Verão 26!274` é um título ("INVERNO 26 (PROVISÓRIO - MARINA) - AINDA SEM ABA PROPRIA"), e os 177 produtos abaixo dele são da coleção Inverno 26. No staging, eles aparecem como aba "Verão 26", e o título virou uma linha de "produto" sem nome nem preço.
- **Por que importa:** a regra da Marina ("vale a coleção mais nova") depende de saber de qual coleção é cada linha.

**SKU em duas abas**
- 154 SKUs: Atemporais + antiga (84), antiga + VERAO 25 (54), Atemporais + Verão 26 (16). 128 com preço diferente. Ex.: `BER003B40` custa 189,90 no Verão 26 e 169,90 no Atemporais.
- **Regras em conflito (decisão de negócio):** Marina (Compras) diz que vale a coleção mais nova; Diego (operação) e o financeiro dizem que vale o menor preço. O mesmo para o custo: último custo (Marina) × custo médio (financeiro).

**Código reaproveitado**
- 6 modelos com mais de um nome. Ex.: `JAQ-001` era "Corta-vento cropped" na planilha antiga e hoje é "Bomber Matelassê"; `MOL-001` era "Moletom careca estonado" e hoje é "Moletom canguru logo". Uma venda antiga com esse código pode ser de outra peça.

**Ligações**
- SKU escrito igual: 55% dos itens de venda acham o produto no catálogo.
- Convertendo os formatos para uma chave comum (CAT + NNN + inicial da cor + tamanho): **77% dos itens** e 74% do estoque. Só um caso de cor ambígua (`ACE-001`: OLI e OFF começam com O).
- Continuam sem par: `DIVERSO` (266), `PRODUTO SEM CADASTRO` (90) e SKUs que não estão em nenhuma aba do catálogo (ex.: `VLT-ACE-008-OLI-U`, 173 itens).
- **Hipótese:** os 23% restantes são peças de coleções que não estão na planilha. Confirmar com a Marina, se for preciso.

**Problemas do catálogo (lista única, nada decidido)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | SKU escrito de 3 jeitos conforme a aba (e o sistema) | todas | ligação com vendas e estoque |
| 2 | SKU em duas abas com preço diferente; regra da Marina × regra do financeiro | 128 | preço migrado |
| 3 | Inverno 26 dentro da aba Verão 26 (+ 1 linha de título) | 177 | coleção de cada produto |
| 4 | Código reaproveitado entre coleções | 6 modelos | vendas antigas ligadas à peça errada |
| 5 | Categoria e cor sem padrão | 57 e 65 grafias | catálogo da Lorenzi |
| 6 | Custo vazio; custo médio × último custo | 198 | custo migrado |
| 7 | Descontinuado sem marcação clara | 93 | o que entra como ativo |
| 8 | EAN repetido em SKUs diferentes | 144 | código de barras |
| 9 | SKUs vendidos que não estão no catálogo | ~23% dos itens | migração de produtos |

**Desafios da revisão**
- **SKUs vendidos fora do catálogo não são peças antigas:** são 452 variantes, e **todas são de modelos que existem no catálogo**; faltam só algumas **cores e tamanhos** (ex.: vendeu o TRI002 na cor G, mas o catálogo só tem a cor B). Foram vendidas igualmente em 2025 e 2026. O catálogo de Compras está incompleto.
- **Preços diferentes entre abas:** a diferença é pequena (mediana de R$ 10, 6,7%; 108 dos 128 abaixo de 10%; um caso acima de 50%). **Em 127 dos 128, a coleção mais nova é a mais cara.** Então as duas regras dão respostas opostas quase sempre: a da Marina (mais nova) leva ao preço maior, e a do Diego/financeiro (menor) ao preço menor. A ordem Atemporais × Verão 26 (16 casos) é discutível.

*(continuar preenchendo ao longo da análise)*